# Lab 42: The Context Cage

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-42.ipynb)

**What you will do:** record your own matched- versus mismatched-room recall counts, weigh them against the book's own sceptical account of context-dependent memory, and watch a small language model "forget" a fact once it is buried outside its fixed context window -- then "remember" it again once it is retrieved.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 42 you studied a 20-word list in one room and were tested in a different room (a
mismatch), then, the next day, studied a new list in the same room and were tested there too
(a match). The book reports that people typically recall modestly more words in the matched
condition, but immediately adds a caveat: the size of this advantage varies a great deal
between studies, and a 2021 replication of the classic diving experiment found no advantage at
all.

## Record your results

Enter your recall counts from both days. `total_words` is how many words were on each list
(20 in the book's version — change it if you used a different length).

In [ ]:
# example data: replace with your own two-day results
total_words = 20
recall_mismatch_A_to_B = 9   # example data: Day 1, studied in Room A, tested in Room B
recall_match_A_to_A = 11     # example data: Day 2 (new list), studied in Room A, tested in Room A

results = pd.DataFrame({
    "condition": ["mismatched (Room A -> Room B)", "matched (Room A -> Room A)"],
    "recalled": [recall_mismatch_A_to_B, recall_match_A_to_A],
    "total": [total_words, total_words],
})
results["proportion recalled"] = (results["recalled"] / results["total"]).round(2)
display(results)

results.set_index("condition")["proportion recalled"].plot.bar(color=["grey", "steelblue"], figsize=(5, 3))
plt.ylim(0, 1); plt.ylabel("proportion recalled"); plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

## Compare

The book is deliberately cautious here: it reports that people "typically" recall "modestly
more" words in the matched condition, but that a 2021 replication of Godden and Baddeley's
original diving experiment found no advantage at all (Murre, 2021), and a meta-analysis
finds environmental context effects to be real but small and inconsistent, sometimes
disappearing altogether (Smith & Vela, 2001). There is no specific percentage in the text to
match your own two data points against — only a direction, and a warning that the direction
itself is unreliable.

In [ ]:
advantage = results.loc[1, "proportion recalled"] - results.loc[0, "proportion recalled"]
print(f"Your matched-minus-mismatched advantage: {advantage:+.0%}")
if advantage > 0:
    print("In the direction the classic effect predicts -- but with a single pair of single-day tests,")
    print("list difficulty, time of day, or ordinary day-to-day variation could easily explain a gap this size.")
elif advantage == 0:
    print("No difference at all -- which is exactly what Murre's 2021 replication found with real divers.")
else:
    print("The mismatched condition won. Per Smith and Vela's meta-analysis, context effects are")
    print("inconsistent enough that a reversal in one small test is unsurprising, not disqualifying.")
print()
print("This is a Lab where the book itself is sceptical of a strong effect, so a modest or absent")
print("advantage in your own data is not a failed replication -- it is close to the expected result.")

## The AI side

The book compares human context-dependent memory with the **context window** of a large
language model: information that has scrolled out of the window is effectively lost, and
**retrieval-augmented generation** (Lewis et al., 2020, *Advances in Neural Information
Processing Systems*) fixes this by fetching the relevant text back. Below, a small open
model is told a fact, which is then buried under a long stretch of unrelated filler text —
enough to push it outside the model's fixed context window — before the model is asked
about it. We then "retrieve" the fact by placing it directly before the question, and
compare how probable the model finds the correct answer in each case.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()
max_context = model.config.n_positions  # distilgpt2's fixed context window, in tokens
print(f"This model can only look back at {max_context} tokens, no matter how long the conversation gets.")

fact_sentence = "By the way, the secret code word for today's meeting is 'lighthouse'."
filler_sentence = "The weather this week has been mild, with a light breeze and clear skies. "
question = " The secret code word for today's meeting is '"
answer = "lighthouse"

def logprob_of_answer(prompt):
    ids_prompt = tokenizer(prompt, return_tensors="pt").input_ids
    ids_full = tokenizer(prompt + answer, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids_full).logits
    logprobs = torch.log_softmax(logits, dim=-1)
    total = 0.0
    for i in range(ids_prompt.shape[1], ids_full.shape[1]):
        total += logprobs[0, i - 1, ids_full[0, i]].item()
    return total

# Condition 1: the fact is stated once, then buried under enough filler to push it out of the window
long_transcript = fact_sentence + " " + filler_sentence * 80 + question
long_ids = tokenizer(long_transcript, return_tensors="pt").input_ids
truncated = tokenizer.decode(long_ids[0, -(max_context - 30):])  # only what the model could still see
buried_lp = logprob_of_answer(truncated)

# Condition 2: "retrieval" -- the fact is fetched back and placed right before the question
retrieved_prompt = fact_sentence + question
retrieved_lp = logprob_of_answer(retrieved_prompt)

ai = pd.DataFrame({"log-prob of the correct code word": [buried_lp, retrieved_lp]},
                   index=["buried under filler text", "retrieved and placed before the question"])
display(ai)

A higher (less negative) log-probability in the "retrieved" row than the "buried" row means
the model is far more confident in the correct code word once the fact is back within reach
— the same fact, made accessible or inaccessible purely by whether it currently sits inside
the context window. This is the AI version of the context cage: nothing about the fact
itself has changed, only whether the current cue can reach it. The disanalogy matters too:
the model's context window has a hard, known size in tokens, while human forgetting from a
room-mismatch is soft and partial rather than all-or-nothing. Do not assume returning to the
original context always restores human memory, though: the book's Beyond the Lab section
describes a closely related "doorway effect" study in which going back to the original room
did *not* bring the forgotten information back, so even the "reopen by returning" version of
the analogy has to be stated cautiously.

## Questions to think about

1. Your two data points differ by only one word list and one room swap. Given Smith and Vela's finding that context effects are "small and inconsistent," how many repeats of this two-day design would you want before trusting a directional conclusion?
2. Nairne (2002) argues that what matters is not whether a cue physically matches the study context, but whether the cue is "diagnostic" -- whether it helps distinguish the target memory from competitors. Is a room a diagnostic cue for a word list, or could almost any consistent room serve equally well? What would make a cue more diagnostic?
3. In the ai_lab demo, the fact is "forgotten" once later text pushes it out of the model's fixed context window. Is this mechanism more like Murre's failed replication (the effect just is not there) or like Godden and Baddeley's original claim (the information is there, but the cue to retrieve it is missing)?
4. The Beyond the Lab section describes the "doorway effect" -- forgetting why you entered a room, because crossing the threshold closes off the previous "event model." Which of your two recall counts, mismatched or matched, is the better analogy for walking through a doorway, and why?

## Challenge

Repeat the two-day design from Steps 1 to 4, but this time keep the room the same and change
your own internal state instead — tired versus alert, sitting versus standing, before versus
after exercise. Compare the size of this state-dependent effect with the room-based
(environmental) effect you measured above.

In [ ]:
# Example data: replace with your own state-dependent memory results.
total_words = 20
recall_state_mismatch = 8     # example data: studied while alert, tested while tired (or vice versa)
recall_state_match = 10       # example data: studied and tested in the same internal state

state_advantage = (recall_state_match - recall_state_mismatch) / total_words
print(f"State-dependent advantage: {state_advantage:+.0%}")
print("Compare the size of this advantage with the room-based (environmental) advantage you measured above.")
print("The text notes that internal states -- mood, arousal, even drug-induced states -- have been proposed")
print("to function as context in a similar way to a physical room, though this state-dependent version of the")
print("effect has proved harder to replicate reliably than the room-based version. The values above are example data.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-42.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")